# O MUNDO DOS ASPIRADORES 

Neste *notebook*, iremos abordar **a estrutura dos agentes** através de um exemplo do **agente aspirador**. O trabalho da IA ​​é conceber um **programa do agente** que implemente a função do agente: o mapeamento das perceções para as ações. Presumimos que este programa será executado num tipo de dispositivo de computação com sensores e atuadores físicos: chamamos a isto **arquitetura**:

<h3 align="center">agente = arquitetura + programa</h3>

## INDICE

* Agente
* Programa Agente Aleatório
* Programa Agente Baseado em Tabela
* Programa Agente Reflexivo Simples
* Programa Agente Reflexivo Baseado em Modelo
* Programa Agente Baseado em Objetivos
* Programa Agente Baseado em Utilidade
* Agente com Aprendizagem

## PROGRAMAS DE AGENTE

Um programa de agente recebe a percepção atual como entrada dos sensores e retorna uma ação aos atuadores. Há uma diferença entre um programa de agente e uma função de agente: um programa de agente toma a percepção atual como entrada, enquanto uma função de agente leva todo o histórico de percepção.

O programa agente toma apenas a percepção atual como entrada porque nada mais está disponível no ambiente; se as ações do agente dependem de toda a sequência de percepção, o agente terá que se lembrar da percepção.

Discutiremos os seguintes programas de agente aqui com a ajuda do exemplo do mundo dos aspiradores:

* Programa de Agente Aleatório
* Programa de Agente Baseado em Tabela
* Programa Agente Reflexivo Simples
* Programa de Agente Reflexio Baseado em Modelo
* Programa de Agente Baseado em Objetivos
* Programa de Agente Baseado em Utilidade

## Programa Agente Aleatório

Um programa de agente aleatório, como o nome sugere, escolhe uma ação de forma aleatória, sem ter em conta as perceções.
Aqui, demonstraremos um agente aspirador aleatório para um ambiente aspirador trivial, ou seja, o ambiente de dois estados.

Vamos importar todas as funções para o módulo do agente:

In [ ]:
from agents import *
from notebook import psource

Vejamos primeiro como definimos o TrivialVacuumEnvironment. Execute a célula seguinte para ver como a classe abstrata TrivialVacuumEnvironment é definida no módulo agents:

In [ ]:
psource(TrivialVacuumEnvironment)

In [3]:
# Estes são os locais triviais no ambiente com apenas dois estados
loc_A, loc_B = (0, 0), (1, 0)

# inicializa o ambiente com dois estados
trivial_vacuum_env = TrivialVacuumEnvironment()

# Verifica o estado do ambiente
print("Estado do Ambiente: {}.".format(trivial_vacuum_env.status))

Estado do Ambiente: {(0, 0): 'Clean', (1, 0): 'Clean'}.


Vamos criar o nosso agente agora. Este agente escolherá qualquer uma das ações entre 'Direita', 'Esquerda', 'Suga' e 'NoOp' (Sem Operação) aleatoriamente.

In [7]:
# Cria o agente aleatório
random_agent = Agent(program=RandomAgentProgram(['Right', 'Left', 'Suck', 'NoOp']))

Agora vamos adicionar o nosso agente ao ambiente.

In [8]:
# Adiciona o agente ao ambiente
trivial_vacuum_env.add_thing(random_agent)

print("O agente aleatório está localizado em {}.".format(random_agent.location))

O agente aleatório está localizado em (0, 0).


Vamos então correr o nosso ambiente.

In [9]:
# Corremos o ambiente um passo
trivial_vacuum_env.step()

# Verificamos o estado do ambiente
print("Estado do Ambiente: {}.".format(trivial_vacuum_env.status))

print("O agente aleatório está localizado em {}.".format(random_agent.location))

Estado do Ambiente: {(0, 0): 'Clean', (1, 0): 'Clean'}.
O agente aleatório está localizado em (1, 0).


## PROGRAMA DE AGENTE BASEADO EM TABELA

Um programa de agente baseado em tabelas monitoriza a sequência de perceções e utiliza-a para indexar numa tabela de ações para decidir o que fazer. A tabela representa explicitamente a função do agente que o programa do agente incorpora.
No mundo do vácuo de dois estados, a tabela seria constituída por todos os estados possíveis do agente.

In [ ]:
table = {((loc_A, 'Clean'),): 'Right',
             ((loc_A, 'Dirty'),): 'Suck',
             ((loc_B, 'Clean'),): 'Left',
             ((loc_B, 'Dirty'),): 'Suck',
             ((loc_A, 'Dirty'), (loc_A, 'Clean')): 'Right',
             ((loc_A, 'Clean'), (loc_B, 'Dirty')): 'Suck',
             ((loc_B, 'Clean'), (loc_A, 'Dirty')): 'Suck',
             ((loc_B, 'Dirty'), (loc_B, 'Clean')): 'Left',
             ((loc_A, 'Dirty'), (loc_A, 'Clean'), (loc_B, 'Dirty')): 'Suck',
             ((loc_B, 'Dirty'), (loc_B, 'Clean'), (loc_A, 'Dirty')): 'Suck'
        }

Vamos agora criar um programa de agente baseado em tabelas para o nosso ambiente de dois estados.

In [ ]:
# Criar o agente baseado em tabelas
table_driven_agent = Agent(program=TableDrivenAgentProgram(table=table))

Como estamos a utilizar o mesmo ambiente, vamos remover o agente aleatório adicionado anteriormente do ambiente para evitar confusão.

In [ ]:
trivial_vacuum_env.delete_thing(random_agent)

In [ ]:
# Adicionar o agente baseado em tabelas ao ambiente
trivial_vacuum_env.add_thing(table_driven_agent)

print("TableDrivenVacuumAgent is located at {}.".format(table_driven_agent.location))

In [ ]:
# Run the environment
trivial_vacuum_env.step()

# Check the current state of the environment
print("State of the Environment: {}.".format(trivial_vacuum_env.status))

print("AgenteBaseadoTabelas está localizado em {}.".format(table_driven_agent.location))

## PROGRAMA DE AGENTE REFLEXIVO SIMPLES

Um programa simples de agente reflexivo seleciona as ações com base na perceção *atual*, ignorando o resto do histórico de perceções. Estes agentes trabalham com uma **regra de condição-ação** (também designada por **regra de situação-ação**, **produção** ou **regra se-então**), que informa o agente da ação a desencadear quando uma situação específica é encontrada.

O esquema esquemático apresentado na figura seguinte tornará isso mais claro:

<img src="./images/agente_simples.png" width="600"/>

Vamos agora criar um agente reflexivo simples para o ambiente.

In [ ]:
# apagamos o agente baseado em tabela adicionada antes
trivial_vacuum_env.delete_thing(table_driven_agent)

Para criar o nosso agente, necessitamos de duas funções: a função INTERPRET-INPUT, que gera uma descrição abstrata do estado atual a partir da perceção, e a função RULE-MATCH, que devolve a primeira regra no conjunto de regras que corresponde à descrição do estado fornecida.

In [ ]:

loc_A = (0, 0)
loc_B = (1, 0)

"""Mudamos o programa do agente reflexivo simples para que acomode a classe Regra"""
def SimpleReflexAgentProgram():
    """Este agente escolhe a ação baseado unicamente na perceção"""
    
    def program(percept):
        loc, status = percept
        return ('Suck' if status == 'Dirty' 
                else'Right' if loc == loc_A 
                            else'Left')
    return program

        
# Cria uma agente reflexivo simple num ambiente de ois estados
program = SimpleReflexAgentProgram()
simple_reflex_agent = Agent(program)

Agora adicione o agente ao ambiente:

In [ ]:
trivial_vacuum_env.add_thing(simple_reflex_agent)

print("SimpleReflexVacuumAgent is located at {}.".format(simple_reflex_agent.location))

In [ ]:
# Corre o ambiente
trivial_vacuum_env.step()

# Verifica o estado corrente do ambiente
print("Estado do ambiente: {}.".format(trivial_vacuum_env.status))

print("O agente reflexivo simples está localizado em {}.".format(simple_reflex_agent.location))

## PROGRAMA DE AGENTE REFLEXiVO BASEADO EM MODELOS

Um agente reflexivo baseado num modelo mantém algum tipo de **estado interno** que depende do histórico de perceção e, portanto, reflete pelo menos alguns dos aspetos não observados do estado atual. Além disso, requer também um **modelo** do mundo, ou seja, esse modelo é um conhecimento sobre "como funciona o mundo".

O esquema esquemático apresentado na figura seguinte tornará isso mais claro:

<img src="images/agente_estado.png" width="600"/>

Vamos agora criar um agente reflexivo baseado em modelo para o ambiente:

In [ ]:
# Removemos o agente reflexivo simples adicionado antes
trivial_vacuum_env.delete_thing(simple_reflex_agent)

Necessitamos de outra função UPDATE-STATE que será responsável por criar uma nova descrição de estado.

In [ ]:
# TODO: Implementar esta função para o ambiente gráfico a duas dimensões
def update_state(state, action, percept, model):
    pass

# Criar o agente reflexivo baseado em modelo
model_based_reflex_agent = ModelBasedVacuumAgent()

# Adicionar os agente ao ambiente
trivial_vacuum_env.add_thing(model_based_reflex_agent)

print("O agente baseado em modelo está localizado em {}.".format(model_based_reflex_agent.location))

In [ ]:
# Correr o ambiente
trivial_vacuum_env.step()

# Verificar o estado corrente do ambiente
print("Estado do ambiente: {}.".format(trivial_vacuum_env.status))

print("O agente baseadoi em modelo está localizado em {}.".format(model_based_reflex_agent.location))

## PROGRAMA DE AGENTE BASEADO EM OBJETIVOS

Um agente baseado em objetivos necessita de algum tipo de informação **objetivo** que descreva situações desejáveis, para além da descrição do estado atual.

A figura seguinte mostra um agente baseado em modelos e em objetivos:

<img src="images/agente_modelo.png" width="600"/>

**Procura** (Capítulos 3 a 5) e **Planeamento** (Capítulos 10 a 11) são os sub àreas da IA ​​dedicadas a encontrar sequências de ações que atinjam os objetivos do agente.

## PROGRAMA DE AGENTE BASEADO NA UTILIDADE

Um agente baseado na utilidade maximiza a sua **utilidade** utilizando a **função de utilidade** do agente, que é essencialmente uma internalização da medida de desempenho do agente.

A figura seguinte mostra um agente baseado em modelo e na utilidade:

<img src="images/agente_utilidade.png" width="600"/>

## AGENTE COM APRENDIZAGEM

A aprendizagem permite que o agente opere em ambientes inicialmente desconhecidos e se torne mais competente do que o seu conhecimento inicial poderia permitir. Aqui, apresentaremos brevemente as principais ideias dos agentes com aprendizagem.

Um agente com aprendizagem pode ser dividido em quatro componentes conceptuais. O **elemento de aprendizagem** é responsável por fazer melhorias. Utiliza o feedback do **crítico** sobre o desempenho do agente e determina como o elemento de desempenho deve ser modificado para melhorar no futuro. O **elemento de desempenho** é responsável pela seleção de ações externas para o agente: recebe perceções e decide sobre ações. O crítico informa o elemento de aprendizagem de quão bem o agente está a desempenhar em relação a um padrão de desempenho fixo. Isto é necessário porque as perceções em si não fornecem qualquer indicação do sucesso do agente. O último componente do agente de aprendizagem é o **gerador de problemas**. É responsável por sugerir ações que levem a experiências novas e informativas.

A figura seguinte resume os componentes e o seu funcionamento:

<img src="images/agente_aprendizagem.png" width="600"/>